# 📦 Prepare YOLO Dataset for Phase 2 Training

Converts the 2000 CATI-extracted samples (with pseudo-labels) into
a standard YOLO dataset structure for Phase 2 fine-tuning.

**No re-labeling needed** — pseudo-labels were saved during feature extraction.

Output structure:
```
yolo_dataset/
  images/train|val|test/*.jpg
  labels/train|val|test/*.txt   ← YOLO format: class cx cy w h
  data.yaml
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, shutil
from pathlib import Path

FEATURE_DIR = '/content/drive/MyDrive/sg_smart_city/data/features'
OUTPUT_DIR  = '/content/drive/MyDrive/sg_smart_city/data/yolo_dataset'

# COCO class_id → CATI class_id
COCO_TO_CATI = {2: 0, 3: 1, 5: 2, 7: 3, 0: 4, 1: 4}
CLASS_NAMES  = ['car', 'motorcycle', 'bus', 'truck', 'person', 'other']
MIN_CONF = 0.3

# Quick sanity check on one JSON to confirm field names
sample = next(Path(FEATURE_DIR).glob('train/*.json'), None)
if sample:
    m = json.loads(sample.read_text())
    print('JSON keys:', list(m.keys()))
    print('source_image:', m.get('source_image', 'MISSING'))
    print('num_detections:', m.get('num_detections'))
else:
    print('❌ No JSON files found — run train_cati.ipynb Cell 6 first')


In [ ]:
stats = {'total': 0, 'copied': 0, 'missing_image': 0, 'with_labels': 0, 'empty': 0}

for split in ['train', 'val', 'test']:
    feat_split = Path(FEATURE_DIR) / split
    if not feat_split.exists():
        print(f'⚠️  {split} not found, skipping')
        continue

    img_out = Path(OUTPUT_DIR) / 'images' / split
    lbl_out = Path(OUTPUT_DIR) / 'labels' / split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    for jf in sorted(feat_split.glob('*.json')):
        stats['total'] += 1
        meta = json.loads(jf.read_text())

        # source_image is the absolute path saved during extraction
        img_src = Path(meta.get('source_image', ''))
        if not img_src.exists():
            stats['missing_image'] += 1
            continue

        # Copy image
        dest_img = img_out / (jf.stem + img_src.suffix)
        if not dest_img.exists():
            shutil.copy(str(img_src), str(dest_img))
        stats['copied'] += 1

        # Write YOLO label
        lines = []
        for det in meta.get('detections', []):
            if det.get('confidence', 0) < MIN_CONF:
                continue
            bbox = det.get('bbox_norm')
            if not bbox or len(bbox) != 4:
                continue
            cati_cls = COCO_TO_CATI.get(det.get('class_id', 0), 5)
            cx, cy, w, h = bbox
            lines.append(f'{cati_cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')

        (lbl_out / (jf.stem + '.txt')).write_text('\n'.join(lines))
        stats['with_labels' if lines else 'empty'] += 1

    imgs = len(list(img_out.glob('*')))
    lbls = len(list(lbl_out.glob('*.txt')))
    print(f'{split}: {imgs} images, {lbls} labels')

print(f'\nStats: {stats}')
print(f'Label coverage: {stats["with_labels"]}/{stats["copied"]} ({100*stats["with_labels"]/max(stats["copied"],1):.1f}%)')


In [ ]:
import yaml

data_yaml = {
    'path': OUTPUT_DIR,
    'train': 'images/train',
    'val':   'images/val',
    'test':  'images/test',
    'nc': 6,
    'names': CLASS_NAMES,
}

yaml_path = Path(OUTPUT_DIR) / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f'✅ data.yaml written')
for split in ["train","val","test"]:
    imgs = len(list((Path(OUTPUT_DIR)/"images"/split).glob("*")))
    lbls = len(list((Path(OUTPUT_DIR)/"labels"/split).glob("*.txt")))
    print(f'  {split}: {imgs} images, {lbls} labels')


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import random

train_lbls = [p for p in (Path(OUTPUT_DIR)/'labels'/'train').glob('*.txt') if p.stat().st_size > 0]
if not train_lbls:
    print('No labeled images found')
else:
    lbl_path = random.choice(train_lbls)
    img_path = next((Path(OUTPUT_DIR)/'images'/'train').glob(lbl_path.stem + '*'))
    img = Image.open(img_path)
    w, h = img.size

    fig, ax = plt.subplots(1, figsize=(10, 6))
    ax.imshow(img)
    colors = ['red','blue','green','orange','purple','gray']
    for line in lbl_path.read_text().strip().split('\n'):
        if not line: continue
        cls, cx, cy, bw, bh = map(float, line.split())
        x1 = (cx - bw/2) * w
        y1 = (cy - bh/2) * h
        rect = patches.Rectangle((x1,y1), bw*w, bh*h, linewidth=2,
                                   edgecolor=colors[int(cls)], facecolor='none')
        ax.add_patch(rect)
        ax.text(x1, y1-5, CLASS_NAMES[int(cls)], color=colors[int(cls)], fontsize=9, fontweight='bold')
    ax.set_title(img_path.name)
    plt.tight_layout(); plt.show()
    print(f'Labels:\n{lbl_path.read_text()}')
